# concurrent.futures

对应 `stdlib.md`：线程池、进程池。

笔记本在 `python_base/concurrent.futures/qa.ipynb`。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_base/concurrent.futures。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "concurrent.futures" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_base" / "concurrent.futures"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_base/concurrent.futures')

## 1. 提交并取回结果

`add(a, b)` 已经写好。用线程池提交 `add(20, 22)`，打印返回值。


In [3]:
def add(a: int, b: int) -> int:
    return a + b

# 作答

import concurrent.futures as futures

with futures.ThreadPoolExecutor() as executor:
    future = executor.submit(add, 20, 22)
    print(future.result())
#评阅
# 对。submit 提交 add(20, 22)，result 取回 42。

#参考答案
# from concurrent.futures import ThreadPoolExecutor
# with ThreadPoolExecutor() as executor:
#     print(executor.submit(add, 20, 22).result())


42


## 2. 按输入顺序收回

`square` 已经写好。用线程池对 `nums` 里每个数求平方，按 `nums` 的顺序打印结果列表。


In [5]:
import time

def square(n: int) -> int:
    time.sleep(0.05 * (5 - n))  # 越小的数越慢
    return n * n

nums = [1, 2, 3, 4]

# 作答

import concurrent.futures as futures

with futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(square, n) for n in nums]
    for future in futures:
        print(future.result())
#评阅
# 顺序对了，但题目要的是一个列表，现在是每行一个数。同一种任务用 map，顺序和输入一致。

#参考答案
# from concurrent.futures import ThreadPoolExecutor
# with ThreadPoolExecutor() as executor:
#     print(list(executor.map(square, nums)))


1
4
9
16


## 3. 谁先做完谁先拿

`work(name, seconds)` 会等待 `seconds` 秒，然后返回 `name`。`jobs` 是要跑的任务。用线程池把它们都提交出去，按完成的先后打印每个任务返回的名字。


In [12]:
import time

def work(name: str, seconds: float) -> str:
    time.sleep(seconds)
    return name

jobs = [("slow", 0.3), ("fast", 0.05), ("mid", 0.15)]

# 作答

import concurrent.futures as futures

with futures.ThreadPoolExecutor() as executor:
    workers = [executor.submit(work, name, seconds) for name, seconds in jobs]
    for result in futures.as_completed(workers):
        print(result.result())
#评阅
# 对。as_completed 按完成先后给出 fast、mid、slow。

#参考答案
# from concurrent.futures import ThreadPoolExecutor, as_completed
# with ThreadPoolExecutor() as executor:
#     workers = [executor.submit(work, name, seconds) for name, seconds in jobs]
#     for future in as_completed(workers):
#         print(future.result())


fast
mid
slow


## 4. 失败的任务

`parse` 已经写好。用线程池提交 `parse("x")`。任务会失败。打印异常类型的名字。


In [13]:
def parse(text: str) -> int:
    return int(text)

# 作答

import concurrent.futures as futures

with futures.ThreadPoolExecutor() as executor:
    future = executor.submit(parse, "x")
    try:
        print(future.result())
    except Exception as e:
        print(type(e).__name__)
#评阅
# 对。result() 把任务里的 ValueError 抛出来，类型名对了。

#参考答案
# from concurrent.futures import ThreadPoolExecutor
# with ThreadPoolExecutor() as executor:
#     future = executor.submit(parse, "x")
#     print(type(future.exception()).__name__)

ValueError


## 5. 换进程池算

用进程池对 `nums` 里每个数求阶乘（`math.factorial`），按 `nums` 的顺序打印结果列表。


In [14]:
import math

nums = [5, 6, 7]

# 作答

import concurrent.futures as futures

with futures.ProcessPoolExecutor() as executor:
    futures = [executor.submit(math.factorial, n) for n in nums]
    for future in futures:
        print(future.result())
#评阅
# 进程池和顺序都对，但题目要打印一个列表，现在是每行一个数。用 map 收成列表。

#参考答案
# from concurrent.futures import ProcessPoolExecutor
# with ProcessPoolExecutor() as executor:
#     print(list(executor.map(math.factorial, nums)))


120
720
5040
